In [1]:
from stock_class import StockList
import pandas as pd
from pathlib import Path
from functools import reduce
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import joblib
import pathlib, datetime, sklearn
from config import ticker_list, bronze, silver, gold, forecast_folder, model_folder, model_meta_file, portfolio_folder
import forecast

In [2]:
test_size  = 250
model_name = f'rf_{datetime.date.today()}'

In [3]:
stock_list = StockList(ticker_list)
stock_list.clear_data()
stock_list.load_data(path=gold)

Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not found. Assigning `Unknown`
Sector not f

In [4]:
X, Y = forecast.create_long_x_y(
    stock_list=stock_list, 
            y_col='log_return', 
            x_cols=None, 
            cols_to_drop=['close', 'high', 'low', 'open', 'ticker', 'sma_9', 'sma_14', 'sma_20', 'sma_50', 'sma_100'], 
            #last_day_only=False
            ) 

In [6]:
dt = datetime.datetime(2024,6,1)
X_train, X_test = X[X.index <= dt], X[X.index > dt]
Y_train, Y_test = Y[Y.index <= dt], Y[Y.index > dt]

X = X_train 
Y = Y_train

In [7]:
pipe = forecast.build_pipeline()
pipe = forecast.fit(pipe=pipe, X=X[X.columns[~X.columns.str.startswith('dt')]], Y=Y)


In [8]:
preds = pipe.predict(X_test[X_test.columns[~X_test.columns.str.startswith('dt')]])

In [9]:
y = pd.DataFrame(Y_test)
y['sign'] = y['target']>0

In [10]:
p = pd.DataFrame({'preds': preds, 'sign_1': preds>0}, index=y.index)

In [11]:
df = pd.concat([p, y], axis=1)

In [12]:
len(df[df['sign'] == df['sign_1']]) / len(df)

0.5147247119078106

In [15]:
df.sort_values('target')

,preds,sign_1,target,sign
2025-04-02,0.002091,True,-0.129399,False
2025-04-02,0.002363,True,-0.107877,False
2025-04-03,0.000049,True,-0.098791,False
2025-04-09,0.000326,True,-0.094120,False
2025-04-03,-0.007562,False,-0.081246,False
...,...,...,...,...
2025-01-14,0.001593,True,0.062879,True
2024-08-14,-0.002176,False,0.065789,True
2024-11-05,0.001834,True,0.080828,True
2025-04-08,-0.008125,False,0.088812,True
